# 영섭: 2-3 분류 시험 + 2-4 감성 점수 (Colab)

**순서** (위에서부터 실행)
1. 설정: 아영님 `articles_clean_100.csv` 경로만 바꾸기
2. 함수 불러오기 (키워드 규칙 v1, 뽑기·일치율·점수 함수)
3. **[1] 100건 뽑기** → 예원님·아영님용 정답지 2개 (컴퓨터 답 없음)
4. **[3] 일치율** → 두 사람 정답을 받은 뒤 실행 (사람끼리 일치율, 컴퓨터 일치율, 화제형 정밀도)
5. **2-4 감성 점수** → 100건으로 먼저 점검, 그다음 전체 → 회사×월 파일

런타임: 4~5번은 **T4 GPU** 권장. 원문(제목·본문)이 들어간 파일은 비공개 드라이브에만 둘 것.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CLEAN = '/content/drive/MyDrive/뉴스_원몬_비공개/articles_clean_100.csv'   # ← 아영님 파일 위치로 바꾸기
OUT   = '/content/drive/MyDrive/4팀_분류시험'                              # 결과 저장 폴더
PREV_TEST = ''   # 지난 시험 100건 id가 든 파일 (있으면 경로, 없으면 빈칸)
import os; os.makedirs(OUT, exist_ok=True); print(os.path.exists(CLEAN))

## 함수 불러오기
아래 두 셀은 고치지 않고 실행만. 키워드는 첫 셀의 `KW` 목록에서 바꿀 수 있음.

In [ ]:
"""2-3 기사 유형 분류: 키워드 규칙 v1 (초안)

판단 근거: 제목 + 본문 앞 200자 (팀 분류 지시문과 같음). 주가·수익률·전망 정보는 판단에 쓰지 않는다.
순서: [기타-시황/리포트 먼저 거름] → 광고 → 공시형 → 논란형 → 화제형 → 기타
복합: 두 유형 이상 키워드가 걸리면 앞 순서로 정하고 복합=Y.
      단, 공시형 + 논란형이 같이 걸리면 논란형 (교수님 예: "유상증자가 논란" → 논란형).
제목에서 먼저 찾고, 제목에 아무것도 없을 때만 본문 앞 200자에서 찾는다.
"""
import re

# 주가·시황·리포트·칼럼: 수익률 정보가 섞이므로 '기타'로 먼저 거른다
OTHER_FIRST = [
    "목표주가", "목표가", "투자의견", "매수 유지", "매수의견", "주가", "시황", "증시", "코스피", "코스닥",
    "특징주", "종목", "펀드", "ETF", "배당주", "수익률", "상한가", "하한가", "신고가", "급등", "급락", "약세", "강세",
    "칼럼", "기고", "사설", "[마켓인사이트]", "HOT종목", "주간", "전망",
    "배당주", "고배당", "투자스토리", "개미", "기관 외인", "株", "보수 1위", "사볼까", "재테크",
]
# 제목이 이 모양이면 증권사 리포트 → 기타  예: "KT&G, ... 시동"-하이투자
REPORT = re.compile(r'"\s*-\s*\S+(증권|투자)\s*$|-\s*\S+증권\s*$')
KW = {
    "광고": [
        "[광고]", "[AD]", "[PR]", "협찬", "기획광고",
        "사회공헌", "기부", "후원", "봉사", "장학", "상생", "기금", "출연", "성금", "나눔", "지원금", "복지",
        "수상", "표창", "시상", "대상 수상", "인증 획득", "인증을 획득", "선정돼", "으뜸기업",
        "공모전", "모집", "채용", "캠페인", "페스티벌", "콘서트", "전시", "공연", "메세나", "문화예술",
        "ESG 경영", "탄소중립", "친환경 활동", "플로깅", "임직원",
        "브랜드 대상", "브랜드경쟁력", "경쟁력지수", "올해의 브랜드", "우수기업", "인증", "소비자 선정", "광고대상",
    ],
    "공시형": [
        "공시", "잠정", "실적", "영업이익", "영업익", "매출", "순이익", "순익", "흑자", "적자", "분기",
        "배당", "자사주", "주주환원", "유상증자", "무상증자", "증자", "감자", "회사채", "채권 발행", "전환사채", "CB 발행", "BW",
        "수주", "공급계약", "단일판매", "계약 체결", "주총", "주주총회", "이사회", "사장 후보", "대표이사 선임", "선임",
        "영업정지", "판매 중단", "사업 중단", "액면분할", "신규 상장", "상장폐지", "합병 결정", "분할 결정",
    ],
    "논란형": [
        "횡령", "배임", "산재", "불매", "소송", "피소", "환경오염", "오염", "압수수색", "검찰", "경찰", "수사", "구속",
        "영장", "기소", "의혹", "과징금", "제재", "적발", "리콜", "결함", "파업", "노조", "갑질", "담합", "유출",
        "사고", "화재", "폭발", "사망", "논란", "고발", "징계", "벌금", "공정위", "부당", "불공정", "비자금", "분식", "탈세",
    ],
    "화제형": [
        "출시", "신제품", "신사업", "론칭", "런칭", "선보", "제휴", "협업", "협약", "MOU", "업무협약", "파트너",
        "진출", "수출", "해외", "법인 설립", "개발", "특허", "기술", "인수", "매각", "합작", "투자 유치", "지분 투자", "설비 투자", "컨소시엄",
        "우선협상", "사업자 선정", "착공", "준공", "가동", "점유율", "돌파", "최초", "확대",
    ],
}
ORDER = ["광고", "공시형", "논란형", "화제형"]


def _hits(text):
    return {t: [k for k in KW[t] if k in text] for t in ORDER}


def classify(title, body=""):
    """반환: (type, 복합 'Y'/'N', 근거 키워드)"""
    title = str(title or ""); body = str(body or "")[:200]
    if REPORT.search(title):
        return "기타", "N", "증권사 리포트"
    strong = ["배당주", "고배당", "펀드", "투자스토리", "개미", "기관 외인", "株", "사볼까", "칼럼"]
    if any(k in title for k in strong):
        return "기타", "N", next(k for k in strong if k in title)
    if any(k in title for k in OTHER_FIRST) and not any(k in title for k in KW["논란형"] + ["공시", "잠정", "실적", "배당", "자사주", "판매 중단", "영업정지"]):
        return "기타", "N", next(k for k in OTHER_FIRST if k in title)
    h = _hits(title)
    if not any(h.values()):
        h = _hits(body)
    got = [t for t in ORDER if h[t]]
    if not got:
        return "기타", "N", ""
    multi = "Y" if len(got) > 1 else "N"
    t = got[0]
    if "공시형" in got and "논란형" in got:      # 공시 사실이 논란의 대상 → 논란형
        t = "논란형"
    if t == "광고" and "논란형" in got:           # 홍보성 문구가 있어도 사건이면 논란형
        t = "논란형"
    return t, multi, h[t][0]


In [ ]:
"""영섭 담당: [1] 100건 뽑기, [3] 일치율, 2-4 감성 점수 — 아영 articles_clean 파일 형식에 자동으로 맞춤"""
import os, re, pandas as pd, numpy as np

CAND = {"id": ["id", "nid", "news_id", "뉴스 식별자", "뉴스식별자"],
        "code": ["code", "종목코드", "stock_code", "ticker"],
        "company": ["company", "회사명", "name", "company_name", "회사"],
        "date": ["date", "일자", "날짜"],
        "outlet": ["outlet", "언론사", "press"],
        "title": ["title", "제목"],
        "body": ["body", "본문", "content"]}


def load_clean(path):
    """아영 articles_clean*.csv → 표준 칸(id, code, company, date, outlet, title, body, year)"""
    d = pd.read_csv(path, dtype=str) if path.endswith(".csv") else pd.read_excel(path, dtype=str)
    out = {}
    for k, cs in CAND.items():
        c = next((c for c in cs if c in d.columns), None)
        out[k] = d[c] if c else pd.Series([""] * len(d))
    x = pd.DataFrame(out)
    if (x.id == "").all():
        x["id"] = [f"r{i}" for i in range(len(x))]
    x["code"] = x.code.str.replace(r"\.0$", "", regex=True).str.zfill(6)
    x["date"] = pd.to_datetime(x.date.str[:10].str.replace(r"^(\d{4})(\d{2})(\d{2})$", r"\1-\2-\3", regex=True), errors="coerce")
    if x.date.isna().all() and "month" in d.columns:      # 날짜 칸이 없으면 month(YYYY-MM)로
        x["date"] = pd.to_datetime(d["month"].str[:7] + "-15", errors="coerce")
    x["year"] = x.date.dt.year
    missing = [k for k in ["code", "date", "title"] if (out[k] == "").all()]
    print(f"읽은 줄 {len(x):,} | 칸 연결: " + ", ".join(f"{k}←{next((c for c in CAND[k] if c in d.columns), '없음')}" for k in CAND))
    if missing:
        print("⚠ 꼭 필요한 칸이 없음:", missing)
    return x


PERIODS = [(2001, 2005), (2006, 2010), (2011, 2015), (2016, 2021)]


def draw_100(x, per_period=25, max_per_company=3, seed=42, exclude_ids=()):
    """기간별 25건, 회사당 최대 3건, 시드 고정. exclude_ids = 지난 시험에 쓴 기사 id"""
    pool = x[~x.id.isin(set(exclude_ids)) & x.year.notna()]
    pool = pool.drop_duplicates(["title", "date"])   # 같은 기사가 여러 회사에 붙은 경우 하나만
    pool = pool.sample(frac=1, random_state=seed)
    picked, cnt = [], {}
    for a, b in PERIODS:
        p = pool[(pool.year >= a) & (pool.year <= b)]
        n = 0
        for i, r in p.iterrows():
            key = r.code if r.code.strip("0") else r.company
            if cnt.get(key, 0) >= max_per_company:
                continue
            picked.append(i); cnt[key] = cnt.get(key, 0) + 1; n += 1
            if n == per_period:
                break
        if n < per_period:
            print(f"⚠ {a}–{b}: {n}건만 뽑힘 (기사 부족)")
    s = pool.loc[picked].copy()
    s.insert(0, "번호", range(1, len(s) + 1))
    print(f"뽑은 건수 {len(s)} | 기간별 {s.groupby(pd.cut(s.year, [2000, 2005, 2010, 2015, 2021])).size().tolist()} | 회사 수 {s.code.nunique()} | 회사당 최대 {s.code.value_counts().max()}")
    return s


def labeling_sheets(s, out_dir, names=("예원", "아영")):
    """사람 정답지: 컴퓨터 답 없이 2부. 열쇠 파일(id 목록)은 따로 저장"""
    os.makedirs(out_dir, exist_ok=True)
    sheet = s[["번호", "id", "company", "code", "date", "outlet", "title"]].copy()
    sheet["본문 앞 200자"] = s.body.str[:200]
    sheet["date"] = sheet.date.dt.strftime("%Y-%m-%d")
    sheet = sheet.rename(columns={"company": "회사", "code": "종목코드", "date": "날짜", "outlet": "언론사", "title": "제목"})
    sheet["정답 (광고/공시형/논란형/화제형/기타)"] = ""; sheet["복합 (Y/N)"] = ""; sheet["메모"] = ""
    paths = []
    for n in names:
        p = f"{out_dir}/분류시험_100_{n}.xlsx"; sheet.to_excel(p, index=False); paths.append(p)
    s.to_csv(f"{out_dir}/분류시험_100_원본.csv", index=False, encoding="utf-8-sig")
    print("정답지:", paths)
    return paths


VALID = ["광고", "공시형", "논란형", "화제형", "기타"]


def _norm(v):
    v = str(v).strip().replace(" ", "")
    for t in VALID:
        if v.startswith(t[:2]):
            return t
    return np.nan


def agreement(path_a, path_b, classify, out_dir, final_col=None):
    """[3] 사람1·사람2·컴퓨터 비교. 사람 둘이 다른 기사는 '최종' 칸을 영섭이 채우고 다시 실행"""
    col = "정답 (광고/공시형/논란형/화제형/기타)"
    a = pd.read_excel(path_a, dtype=str); b = pd.read_excel(path_b, dtype=str)
    m = a[["번호", "id", "회사", "제목", "본문 앞 200자", col, "복합 (Y/N)"]].merge(
        b[["번호", col, "복합 (Y/N)"]], on="번호", suffixes=("_A", "_B"))
    m["사람A"] = m[f"{col}_A"].map(_norm); m["사람B"] = m[f"{col}_B"].map(_norm)
    hh = (m.사람A == m.사람B).mean()
    m["최종"] = np.where(m.사람A == m.사람B, m.사람A, np.nan)
    fp = f"{out_dir}/분류시험_100_최종정답.xlsx"
    if os.path.exists(fp):                      # 영섭이 채운 최종 정답 반영
        f = pd.read_excel(fp, dtype=str).set_index("번호")["최종"].map(_norm)
        m["최종"] = m.번호.map(f).fillna(m["최종"])
    r = m.apply(lambda x: classify(x["제목"], x["본문 앞 200자"]), axis=1, result_type="expand")
    m[["컴퓨터", "컴퓨터_복합", "근거"]] = r
    ok = m.최종.notna()
    acc = (m.loc[ok, "컴퓨터"] == m.loc[ok, "최종"]).mean()
    tp = m[ok & (m.컴퓨터 == "화제형")]; tr = m[ok & (m.최종 == "화제형")]
    prec = (tp.최종 == "화제형").mean() if len(tp) else np.nan
    rec = (tr.컴퓨터 == "화제형").mean() if len(tr) else np.nan
    print(f"사람끼리 일치 {hh:.0%} ({(m.사람A == m.사람B).sum()}/{len(m)})")
    print(f"최종 정답이 정해진 기사 {ok.sum()}건 | 컴퓨터 일치율 {acc:.0%} (목표 80%) | 화제형 정밀도 {prec:.0%}, 재현율 {rec:.0%}")
    print(pd.crosstab(m.loc[ok, "최종"], m.loc[ok, "컴퓨터"], rownames=["사람 최종"], colnames=["컴퓨터"], margins=True))
    if not os.path.exists(fp):
        m[["번호", "회사", "제목", "사람A", "사람B", "최종"]].to_excel(fp, index=False)
        print(f"→ 사람 둘이 다른 {(~ok).sum()}건: {fp} 의 '최종' 칸을 채우고 이 셀을 다시 실행")
    m[ok & (m.컴퓨터 != m.최종)][["번호", "회사", "제목", "최종", "컴퓨터", "근거"]].to_excel(f"{out_dir}/분류시험_100_틀린것.xlsx", index=False)
    return m


# ---------- 2-4 감성 점수 ----------
def load_finbert(model="snunlp/KR-FinBert-SC"):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(model)
    mdl = AutoModelForSequenceClassification.from_pretrained(model).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; mdl.to(dev)
    if dev == "cuda": mdl.half()
    lab = {v.lower(): k for k, v in mdl.config.id2label.items()}
    print("모델", model, "| 장치", dev, "| 라벨", mdl.config.id2label)
    return tok, mdl, dev, lab


def score(texts, fb, bs=128, max_len=160):
    """s = P(긍정) − P(부정), −1~+1. 재학습 없음"""
    import torch
    tok, mdl, dev, lab = fb; out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(dev)
        with torch.no_grad():
            p = torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy()
        out.append(p[:, lab["positive"]] - p[:, lab["negative"]])
    return np.concatenate(out) if out else np.array([])


def assign_month(dates):
    """마지막 거래일(평일 근사) 당일·이후 기사는 다음 달로"""
    per = dates.dt.to_period("M")
    last = (dates + pd.offsets.BMonthEnd(0)).dt.normalize()
    return per.where(dates < last, per + 1).astype(str)


def monthly(x):
    """회사×월 집계 (원문 없음 → 밖으로 내보내도 됨)"""
    g = x.groupby(["month", "code"])
    out = g.size().rename("n_all").to_frame()
    for t, k in [("화제형", "topic"), ("공시형", "disc"), ("논란형", "contro"), ("광고", "ad")]:
        sub = x[x.type == t].groupby(["month", "code"])
        out[f"n_{k}"] = sub.size(); out[f"{k}_tone"] = sub.s.mean()
    for c in [c for c in out if c.startswith("n_")]:
        out[c] = out[c].fillna(0).astype(int)
    return out.reset_index()


## [1] 100건 뽑기
기간 4구간(2001–05 / 06–10 / 11–15 / 16–21) × 25건, 회사당 최대 3건, seed 42. 지난 시험 기사는 빼고 뽑음.

In [ ]:
x = load_clean(CLEAN)
prev = []
if PREV_TEST:
    p = pd.read_csv(PREV_TEST, dtype=str) if PREV_TEST.endswith('.csv') else pd.read_excel(PREV_TEST, dtype=str)
    prev = p[[c for c in p.columns if c in ('id', '뉴스 식별자')][0]].tolist()
s = draw_100(x, per_period=25, max_per_company=3, seed=42, exclude_ids=prev)
paths = labeling_sheets(s, OUT, names=('예원', '아영'))
# → 두 파일을 예원님, 아영님께 각각 전달 (서로 보지 않고 '정답' 칸 채우기, 헷갈리면 복합=Y)

## [3] 일치율 (두 사람 정답을 받은 뒤)
사람 둘이 다르게 단 기사는 `분류시험_100_최종정답.xlsx`의 '최종' 칸을 영섭이 채우고 이 셀을 다시 실행.

In [ ]:
m = agreement(f'{OUT}/분류시험_100_예원.xlsx', f'{OUT}/분류시험_100_아영.xlsx', classify, OUT)
# 틀린 기사 목록: 분류시험_100_틀린것.xlsx → 키워드를 고친 뒤에는 '새' 100건(seed 다르게)으로 다시 재기

## 2-4 감성 점수 — ① 100건으로 점검
KR-FinBert-SC, 재학습 없음. s = P(긍정) − P(부정). 가장 긍정·부정 10건을 눈으로 보고 방향이 맞는지 확인.

In [ ]:
fb = load_finbert()
s['s'] = score((s.title.fillna('') + ' ' + s.body.fillna('').str[:200]).tolist(), fb)
view = s[['company', 'title', 's']].sort_values('s')
print('가장 부정 10건'); print(view.head(10).to_string(index=False))
print('가장 긍정 10건'); print(view.tail(10).to_string(index=False))

## 2-4 감성 점수 — ② 전체 → 회사×월
모든 기사 분류 → 광고 제외 점수 → 월 배정(마지막 거래일 이후는 다음 달) → 집계. 결과 파일에는 원문이 없음.

In [ ]:
x[['type', '복합', '근거']] = x.apply(lambda r: classify(r.title, r.body), axis=1, result_type='expand')
print(x.type.value_counts(normalize=True).round(3))
use = x[x.type != '광고'].copy()
import time; t0 = time.time()
use['s'] = score((use.title.fillna('') + ' ' + use.body.fillna('').str[:200]).tolist(), fb)
print(f'점수 {len(use):,}건, {time.time()-t0:.0f}초 → 1만 건당 {10000*(time.time()-t0)/max(len(use),1):.0f}초')
x = x.merge(use[['id', 's']], on='id', how='left')
x['month'] = assign_month(x.date)
x[['id', 'code', 'date', 'month', 'type', '복합', 's']].to_csv(f'{OUT}/article_scores_100.csv', index=False, encoding='utf-8-sig')
mon = monthly(x); mon.to_csv(f'{OUT}/news_monthly_100.csv', index=False, encoding='utf-8-sig')
print(mon.shape); print(mon.describe().round(3))